In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelConfig:
    model_name: str
    encoder:    str
    encoder_weights:    str
    checkpoint_dir:     Path
    latest_model_dir:   Path
    best_model_dir:     Path
    run_info_dir:       Path

@dataclass(frozen=True)
class DataConfig:
    train_data_dir: Path
    valid_data_dir: Path

@dataclass(frozen=True)
class OptimizerConfig:
    lr:             float
    decay:          float
    lr_scheduler:   str

@dataclass(frozen=True)
class MetricConfig:
    metric_mode:    str
    loss_function:  str
    alpha:          float
    gamma:          float

@dataclass(frozen=True)
class TrainParamsConfig:
    batch_size:     int
    patience:       int
    start_epoch:    int
    epochs:         int
    workers:        int
    seed:           int
    image_size:     int
    is_augmentation: bool

@dataclass(frozen=True)
class TrainingConfig:
    root_dir:   Path
    model:      ModelConfig
    data:       DataConfig
    metric:     MetricConfig
    optimizer:  OptimizerConfig
    train:      TrainParamsConfig

In [6]:
from pneumonia_segmentation.constants import *
from pneumonia_segmentation.utils.common import read_yaml, create_directories

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    # ── private helpers ───────────────────────────────────────
    def _model_slug(self) -> str:
        p = self.params.prepare_base_model_params
        return f"{p.model_name}_{p.encoder}"
    
    def _get_model_config(self, train_root: Path) -> ModelConfig:
        p    = self.params.prepare_base_model_params
        slug = self._model_slug()
        d    = train_root / slug
        os.makedirs(d, exist_ok=True)
        
        return ModelConfig(
            model_name      = p.model_name,
            encoder         = p.encoder,
            encoder_weights = p.encoder_weights,
            checkpoint_dir  = d / "checkpoints",
            latest_model_dir= d / "model.pth",
            best_model_dir  = d / "best_model.pth",
            run_info_dir    = d / "run_info.json"
            
        )
    
    def _get_data_config(self) -> DataConfig:
        c = self.config.data_transformation_config
        return DataConfig(
            train_data_dir  = Path(c.out_train_dir),
            valid_data_dir  = Path(c.out_valid_dir),  
        )
    
    def _get_optimizer_config(self) -> OptimizerConfig:
        p = self.params.training_params
        return OptimizerConfig(
            lr = p.lr,
            decay = p.decay,
            lr_scheduler = p.lr_scheduler
        )
    
    def _get_metric_config(self) -> MetricConfig:
        p = self.params.training_params
        return MetricConfig(
            metric_mode     = p.metric_mode,
            loss_function   = p.loss_function,
            alpha = p.alpha,
            gamma = p.gamma
        )
    
    def _get_train_params_config(self) -> TrainParamsConfig:
        p = self.params.training_params
        return TrainParamsConfig(
            batch_size      = p.batch_size,
            patience        = p.patience,
            start_epoch     = p.start_epoch,
            epochs          = p.epochs,
            workers         = p.workers,
            seed            = p.seed,
            image_size      = p.image_size,
            is_augmentation = p.is_augmentation,
        )

    # ── public ───────────────────────────────────────────────
    def get_training_config(self) -> TrainingConfig:
        train_root = Path(self.config.training_config.root_dir)
        create_directories([train_root])

        return TrainingConfig(
            root_dir  = train_root,
            model     = self._get_model_config(train_root),
            data      = self._get_data_config(),
            metric    = self._get_metric_config(),
            optimizer = self._get_optimizer_config(),
            train     = self._get_train_params_config(),
        )

In [ ]:
import os, sys, time, torch, json
import torch.nn as nn
import segmentation_models_pytorch as smp

from pneumonia_segmentation import logging
from pneumonia_segmentation.exception import CustomException

from pneumonia_segmentation.utils.data_loaders import get_dataloaders
from pneumonia_segmentation.utils.helpers.optimizer import Optimizer
from pneumonia_segmentation.utils.helpers.early_stopper import EarlyStopper
from pneumonia_segmentation.utils.helpers.lr_scheduler import LR_Scheduler
from pneumonia_segmentation.utils.engine.engine import train_one_epoch, validate

MODEL_MAP = {
    "unet":        smp.Unet,
    "unetpp":      smp.UnetPlusPlus,
    "deeplabv3":   smp.DeepLabV3,
    "segformer":   smp.Segformer,
    "manet":       smp.MAnet,
}

class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.best_iou = 0.0 if config.metric.metric_mode == "max" else float('inf')
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        self.loaders        = get_dataloaders(self.config)
        self.model          = self._get_model()
        self.optimizer      = Optimizer().get_optim(self.model, self.config)
        self.loss_function  = self._get_loss_function()
        self.early_stopper  = EarlyStopper(
            patience    = self.config.train.patience, 
            mode        = self.config.metric.metric_mode
        )
        self.scheduler      = LR_Scheduler.get_lr_scheduler(
            config      = self.config, 
            loaders     = self.loaders, 
            optimizer   = self.optimizer
        )
    
    def _get_model(self) -> nn.Module:
        model_name = self.config.model.model_name.lower()
        if model_name not in MODEL_MAP:
            raise ValueError(
                f"Model '{model_name}' not supported. Choose from {list(MODEL_MAP.keys())}"
            )
        model = MODEL_MAP[model_name](
            encoder_name    = self.config.model.encoder,
            encoder_weights = self.config.model.encoder_weights,
            classes         = 1,
            activation      = None,
        )
        return model.to(self.device)

    def _get_loss_function(self):
        dice = smp.losses.DiceLoss(mode='binary')
        focal = smp.losses.FocalLoss(mode='binary', alpha=self.config.metric.alpha, gamma=self.config.metric.gamma)
        return lambda preds, masks: dice(preds, masks) + focal(preds, masks)
    
    # ── training loop ─────────────────────────────────────────
    
    def train(self):
        for epoch in range(self.config.train.start_epoch, self.config.train.epochs):
            start = time.time()
            train_loss = train_one_epoch(
                self.model, self.loaders['train'], 
                self.loss_function, self.optimizer, 
                self.scheduler, epoch, self.device
            )
            
            if epoch % 2 == 0:
                valid_loss, mean_iou = self._run_validation(epoch)
                elapsed = time.time() - start
                self._log_epoch(epoch, elapsed, train_loss, valid_loss, mean_iou)

                score = -0.25*train_loss -0.25*valid_loss + 0.5*mean_iou
                self.early_stopper(score)
                if self.early_stopper.early_stop:
                    logging.info("Early stopping triggered.")
                    break

        torch.save(self.model.state_dict(), self.config.model.latest_model_dir)
        self.save_run_info()
    
    # ── validation ────────────────────────────────────────────
    
    def _run_validation(self, epoch) -> tuple[float, float]:
        valid_loss, mean_iou = validate(self.model, self.loaders['valid'], self.loss_function, epoch, self.device)
        lr = self.scheduler.get_last_lr()[0] if self.config.optimizer.lr_scheduler is not None else self.config.optimizer.lr
        
        self._update_best(mean_iou)
        self.save_checkpoint(epoch, lr, mean_iou)
        return valid_loss, mean_iou
    
    def _update_best(self, mean_iou):
        if self.config.metric.metric_mode == "max":
            is_better = mean_iou > self.best_iou
            self.best_iou = max(mean_iou, self.best_iou)
        else:
            is_better = mean_iou < self.best_iou
            self.best_iou = min(mean_iou, self.best_iou)
        self.save_best_model(is_better)
    
    # ── save helpers ─────────────────────────────────────────
    
    def save_checkpoint(self, epoch: int, lr: float, metric: float, keep_last: int = 5):
        ckpt_dir = self.config.model.checkpoint_dir
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        
        torch.save(
            {
                "epoch":                epoch,
                "model_state_dict":     self.model.state_dict(),
                "optimizer_state_dict": self.optimizer.state_dict(),
                "lr":                   lr,
                "metric":               metric,
            },
            ckpt_dir / f"checkpoint_epoch_{epoch}.pth",
        )

        for old_ckpt in sorted(ckpt_dir.glob("checkpoint_epoch_*.pth"))[:-keep_last]:
            old_ckpt.unlink()
    
    def save_best_model(self, is_better: bool):
        if is_better:
            torch.save(
                self.model.state_dict(), 
                self.config.model.best_model_dir
            )
        logging.info(f"Best model updated — IOU: {self.best_iou:.4f}")
        
    def save_run_info(self):
        m = self.config.model
        m.run_info_dir.parent.mkdir(parents=True, exist_ok=True)
        
        run_info = {
            "model_name": m.model_name,
            "encoder":    m.encoder,
            "iou_score":  self.best_iou,
            "status":     "success",
        }
        
        with open(m.run_info_dir, "w") as f:
            json.dump(run_info, f, indent=4)
        logging.info(f"Run info saved at {m.run_info_dir}")
        
    # ── logging ──────────────────────────────────────────────
    
    def _log_epoch(self, epoch: int, elapsed: float, train_loss, valid_loss, mean_iou):
        logging.info(
            f"Epoch {epoch} | {elapsed / 60:.2f} min | "
            f"Train Loss {train_loss:.4f} | "
            f"Valid Loss {valid_loss:.4f} | "
            f"IOU {mean_iou:.4f} | Best {self.best_iou:.4f}"
        )

In [12]:
try:
    config = ConfigurationManger()
    trainer = Training(config=config.get_training_config())
    trainer.train()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-05 23:39:34,983: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-05 23:39:34,988: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-05 23:39:34,989: INFO: common: created directory at: artifacts]
[2026-04-05 23:39:34,991: INFO: common: created directory at: artifacts\training]


[2026-04-05 23:39:47,626: INFO: 2718912002: Best model updated — IOU: 0.0525]
[2026-04-05 23:39:47,999: INFO: 2718912002: Epoch 0 | 0.21 min | Train Loss 1.2023 | Valid Loss 1.5700 | IOU 0.0525 | Best 0.0525]


[2026-04-05 23:40:10,519: INFO: 2718912002: Best model updated — IOU: 0.3782]
[2026-04-05 23:40:10,856: INFO: 2718912002: Epoch 2 | 0.19 min | Train Loss 0.9087 | Valid Loss 0.8564 | IOU 0.3782 | Best 0.3782]


[2026-04-05 23:40:32,913: INFO: 2718912002: Best model updated — IOU: 0.4849]
[2026-04-05 23:40:33,249: INFO: 2718912002: Epoch 4 | 0.19 min | Train Loss 0.8335 | Valid Loss 0.8014 | IOU 0.4849 | Best 0.4849]


[2026-04-05 23:40:56,046: INFO: 2718912002: Best model updated — IOU: 0.5438]
[2026-04-05 23:40:56,519: INFO: 2718912002: Epoch 6 | 0.21 min | Train Loss 0.7835 | Valid Loss 0.7644 | IOU 0.5438 | Best 0.5438]


[2026-04-05 23:41:19,338: INFO: 2718912002: Best model updated — IOU: 0.5978]
[2026-04-05 23:41:19,857: INFO: 2718912002: Epoch 8 | 0.20 min | Train Loss 0.7566 | Valid Loss 0.7442 | IOU 0.5978 | Best 0.5978]


[2026-04-05 23:41:30,564: INFO: 2718912002: Run info saved at artifacts\training\unetpp_efficientnet-b3\run_info.json]
